# RAG evaluation (answer quality + latency)

This notebook loads questions from the CSVs in `notebooks/` and (optionally) ingests all `.txt` files from `rag-dataset/rag-dataset/`. It then measures:

- **Answer quality** — LLM-as-Judge scores (faithfulness, relevance, completeness, citation accuracy, overall) via `EvaluationService`.
- **Latency** — retrieval time, streaming TTFT (time to first token), generation wall time, end-to-end time. Reports **p50** and **p95**.

**Prerequisites**

- Run from the `backend/notebooks` directory (or keep paths below correct).
- `.env` configured (Postgres, Chroma, embedding + LLM + evaluator API keys).
- For optional HTTP ingest: Flask API running (e.g. `localhost:5000`) with Chroma + DB up.
- Alternatively set `COLLECTION_ID` to an existing collection that already contains the rag-dataset texts.
- Use the **Inspect collections** cell (`GET /api/retrieval/stats/<id>`) to print chunk counts and document filenames for one or more collection ids before you pick `COLLECTION_ID`.

**Note:** Without labeled relevant chunks, retrieval **Recall@k** is not computed here.

In [1]:
# --- Configuration ---
import os
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks":
    # Allow running from backend/ root
    _nb = NOTEBOOK_DIR / "notebooks"
    if _nb.is_dir():
        NOTEBOOK_DIR = _nb.resolve()

BACKEND_ROOT = NOTEBOOK_DIR.parent
RAG_DATASET_DIR = NOTEBOOK_DIR / "rag-dataset" / "rag-dataset"

# Set via env EVAL_COLLECTION_ID, or uncomment and set an id to skip ingest:
COLLECTION_ID = os.environ.get("EVAL_COLLECTION_ID")
COLLECTION_ID = int(COLLECTION_ID) if COLLECTION_ID else None
COLLECTION_ID = 11

# "http" = create collection + upload via API (requires Flask on API_BASE). "skip" = use COLLECTION_ID only
INGEST_MODE = os.environ.get("EVAL_INGEST_MODE", "skip")
API_BASE = os.environ.get("EVAL_API_BASE", "http://localhost:5000")
GUEST_SESSION_ID = os.environ.get("EVAL_GUEST_SESSION_ID", "evaluationnotebooksessionid")

# Cap rows per split for faster runs (None = all rows)
MAX_QUESTIONS = int(os.environ["EVAL_MAX_QUESTIONS"]) if os.environ.get("EVAL_MAX_QUESTIONS") else None

# Answer generator (RAG)
ANSWER_PROVIDER = os.environ.get("EVAL_ANSWER_PROVIDER")  # None -> Config default
ANSWER_MODEL = os.environ.get("EVAL_ANSWER_MODEL")  # None -> Config default

# Judge uses Config.EVALUATOR_* unless overridden
EVAL_PROVIDER = os.environ.get("EVAL_EVALUATOR_PROVIDER")
EVAL_MODEL = os.environ.get("EVAL_EVALUATOR_MODEL")

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("RAG_DATASET_DIR:", RAG_DATASET_DIR, "exists:", RAG_DATASET_DIR.is_dir())
print("COLLECTION_ID:", COLLECTION_ID, "INGEST_MODE:", INGEST_MODE)

NOTEBOOK_DIR: C:\Users\Devendra\opensource\ai-research-assistant\backend\notebooks
RAG_DATASET_DIR: C:\Users\Devendra\opensource\ai-research-assistant\backend\notebooks\rag-dataset\rag-dataset exists: True
COLLECTION_ID: 11 INGEST_MODE: skip


In [4]:
# Inspect existing collections via GET /api/retrieval/stats/<id> (backend: get_collection_stats)
# Requires: run the config cell first (API_BASE, GUEST_SESSION_ID). Flask API must be running.
import os
import requests

# Set ids here OR set env EVAL_INSPECT_COLLECTION_IDS=11,15,22
_raw = os.environ.get("EVAL_INSPECT_COLLECTION_IDS", "")
COLLECTION_IDS_TO_INSPECT = (
    [int(x.strip()) for x in _raw.split(",") if x.strip().isdigit()]
    if _raw.strip()
    else []
)
# If env is empty, fill your collection ids below:
if not COLLECTION_IDS_TO_INSPECT:
    COLLECTION_IDS_TO_INSPECT = [11, 15]

headers = {"GuestUserSessionId": GUEST_SESSION_ID}

for cid in COLLECTION_IDS_TO_INSPECT:
    url = f"{API_BASE}/api/retrieval/stats/{cid}"
    try:
        r = requests.get(url, headers=headers, timeout=60)
        if r.status_code == 404:
            print(f"collection_id={cid}: not found or not accessible for this guest session")
            continue
        r.raise_for_status()
        data = r.json()
        print(f"--- collection_id={cid} ---")
        print(f"  chunks_count: {data.get('chunks_count')}")
        docs = data.get("documents") or []
        print(f"  documents ({len(docs)}):")
        for name in sorted(docs):
            print(f"    - {name}")
    except requests.RequestException as e:
        print(f"collection_id={cid}: request failed: {e}")


--- collection_id=11 ---
  chunks_count: 986
  documents (20):
    - alanwake.fandom.com_wiki_Alan_Wake_2.txt
    - arxiv.org_pdf_2404.10981.txt
    - bg3.wiki_wiki_The_Emperor.txt
    - blog.reedsy.com_short-story_a3gstd.txt
    - bytes-and-nibbles.web.app_bytes_stici-note-part-1-planning-and-prototyping.txt
    - dmtalkies.com_the-zone-of-interest-ending-explained-and-summary-2023-film.txt
    - docs.marimo.io_recipes.html.txt
    - ec.europa.eu_commission_presscorner_detail_en_QANDA_21_1683.txt
    - enterthegungeon.fandom.com_wiki_Bullet_Kin.txt
    - github.com_llmware-ai_llmware.txt
    - gleam.run_cheatsheets_gleam-for-python-users.txt
    - stardewvalleywiki.com_Version_History.txt
    - timdettmers.com_2023_01_30_which-gpu-for-deep-learning.txt
    - towardsdatascience.com_gpt-from-scratch-with-mlx-acf2defda30e.txt
    - towardsdatascience.com_how-to-maximize-your-impact-as-a-data-scientist-3881995a9cb1.txt
    - whattocook.substack.com_p_so-into-northern-spain.txt
    - www.c

In [2]:
import sys
import time
import json
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

sys.path.insert(0, str(BACKEND_ROOT))

from app import create_app
from app.config import Config

app = create_app()
_app_ctx = app.app_context()
_app_ctx.push()
print("App context active. Default LLM:", Config.DEFAULT_LLM_PROVIDER, Config.DEFAULT_MODEL_NAME)

c:\Users\Devendra\opensource\ai-research-assistant\backend\venv312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
c:\Users\Devendra\opensource\ai-research-assistant\backend\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-25 19:56:05,055 [INFO] botocore.credentials: Found credentials in environment variables.
2026-03-25 19:56:05,561 [INFO] app.services.embedding_service: loading the HuggingFaceEmbeddings model: BAAI/bge-small-en-v1.5 with normalize_embeddings set to True
2026-03-25 19:56:05,564 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: cuda:0
2026-03-25 19:56:05,566 [INFO] sentence_transformers.SentenceTransformer: Load pretrained 

App context active. Default LLM: ollama llama3.2


In [3]:
def ingest_via_http() -> int:
    """Create collection and upload all .txt files from RAG_DATASET_DIR. Returns collection_id."""
    import requests

    headers = {"GuestUserSessionId": GUEST_SESSION_ID}
    r = requests.post(
        f"{API_BASE}/api/collections",
        json={"name": "notebook_eval_rag_dataset"},
        headers=headers,
        timeout=60,
    )
    r.raise_for_status()
    cid = r.json()["id"]
    txt_files = sorted(RAG_DATASET_DIR.glob("*.txt"))
    if not txt_files:
        raise FileNotFoundError(f"No .txt files under {RAG_DATASET_DIR}")
    for i, path in enumerate(txt_files):
        with open(path, "rb") as f:
            requests.post(
                f"{API_BASE}/api/documents/upload/{cid}",
                files={"file": (path.name, f, "text/plain")},
                headers=headers,
                timeout=600,
            ).raise_for_status()
        print(f"Uploaded ({i+1}/{len(txt_files)}): {path.name}")
        time.sleep(2)
    return cid


if COLLECTION_ID is None:
    if INGEST_MODE == "http":
        COLLECTION_ID = ingest_via_http()
    else:
        raise ValueError("Set COLLECTION_ID or INGEST_MODE=http with API running.")
else:
    print("Using existing COLLECTION_ID:", COLLECTION_ID)

print("Eval collection_id:", COLLECTION_ID)

Using existing COLLECTION_ID: 11
Eval collection_id: 11


In [4]:
import pandas as pd
from typing import Optional


def load_question_frames(max_rows: Optional[int]) -> pd.DataFrame:
    rows = []
    single = pd.read_csv(NOTEBOOK_DIR / "single_passage_answer_questions.csv")
    single["split"] = "single_passage"
    rows.append(single)
    multi = pd.read_csv(NOTEBOOK_DIR / "multi_passage_answer_questions.csv")
    multi["split"] = "multi_passage"
    rows.append(multi)
    no_ans = pd.read_csv(NOTEBOOK_DIR / "no_answer_questions.csv")
    no_ans["split"] = "no_answer"
    if "answer" not in no_ans.columns:
        no_ans["answer"] = float("nan")
    rows.append(no_ans)
    df = pd.concat(rows, ignore_index=True)
    if max_rows is not None:
        df = df.head(max_rows)
    return df


questions_df = load_question_frames(MAX_QUESTIONS)
print(len(questions_df), "questions loaded")
questions_df.head()

120 questions loaded


,document_index,question,answer,split
0,0,What do keybullet kin drop?,Keybullet kin drop a key upon death.,single_passage
1,0,What kind of gun does the bandana bullet kin use?,The bandana bullet kin wields a machine pistol.,single_passage
2,1,What do the giants look like?,"One giant is burly, grey-skinned, and 20 feet ...",single_passage
3,1,What happens on day 2?,"After a few miles of winding tunnel, you emerg...",single_passage
4,2,What were the requirements for the project?,The tool had the following requirements:\r\n- ...,single_passage


In [6]:
def percentile_ms(values: List[float], q: float) -> float:
    arr = np.array([x for x in values if x is not None and not np.isnan(x)], dtype=float)
    if arr.size == 0:
        return float("nan")
    return float(np.percentile(arr, q))


def summarize_timings(name: str, values: List[float]) -> None:
    arr = np.array([x for x in values if x is not None and not np.isnan(x)], dtype=float)
    if arr.size == 0:
        print(f"{name}: no samples")
        return
    print(
        f"{name}: n={arr.size} mean={arr.mean():.1f}ms p50={percentile_ms(values, 50):.1f}ms "
        f"p95={percentile_ms(values, 95):.1f}ms max={arr.max():.1f}ms"
    )


def run_one_question(
    rag: Any,
    collection_id: int,
    question: str,
    answer_provider: Optional[str],
    answer_model: Optional[str],
) -> Tuple[Dict[str, Any], Dict[str, float]]:
    """Retrieve, stream-generate (for TTFT), evaluate. Returns (eval_dict, timings_ms)."""
    t0 = time.perf_counter()
    chunks = rag.retriever.retrieve(collection_id=collection_id, query=question)
    t1 = time.perf_counter()
    messages = rag._build_messages(question=question, chunks=chunks, conversation_id=None)

    first_token_t: Optional[float] = None
    buf: List[str] = []
    for token in rag.llm_service.generate_stream(
        messages=messages,
        provider=answer_provider,
        model_name=answer_model,
    ):
        if first_token_t is None:
            first_token_t = time.perf_counter()
        buf.append(token)
    t2 = time.perf_counter()
    answer = "".join(buf)

    retrieval_ms = (t1 - t0) * 1000.0
    e2e_ms = (t2 - t0) * 1000.0
    gen_wall_ms = (t2 - t1) * 1000.0
    ttft_ms = (first_token_t - t0) * 1000.0 if first_token_t else float("nan")
    ttft_after_retrieval_ms = (first_token_t - t1) * 1000.0 if first_token_t else float("nan")

    timings = {
        "retrieval_ms": retrieval_ms,
        "ttft_ms": ttft_ms,
        "ttft_after_retrieval_ms": ttft_after_retrieval_ms,
        "generation_wall_ms": gen_wall_ms,
        "e2e_ms": e2e_ms,
    }

    evaluation = rag.evaluate_response(
        question=question,
        answer=answer,
        chunks=chunks,
        provider=EVAL_PROVIDER,
        model_name=EVAL_MODEL,
    )
    return evaluation, timings


from app.services.rag_pipeline import RAGPipeline

rag = RAGPipeline()

results: List[Dict[str, Any]] = []
n_total = len(questions_df)
for n, (idx, row) in enumerate(questions_df.iterrows(), start=1):
    q = str(row["question"]).strip()
    split = row.get("split", "")
    try:
        ev, timings = run_one_question(
            rag,
            COLLECTION_ID,
            q,
            ANSWER_PROVIDER,
            ANSWER_MODEL,
        )
        row_out = {
            "idx": int(idx),
            "split": split,
            "question": q[:200],
            "evaluation": ev,
            "timings_ms": timings,
            "error": ev.get("error"),
        }
    except Exception as e:
        row_out = {"idx": int(idx), "split": split, "question": q[:200], "error": str(e)}
    results.append(row_out)
    if n % 5 == 0 or n == n_total:
        print(f"Processed {n}/{n_total}")
        break

print("Done.")

2026-03-25 20:04:37,323 [INFO] app.services.bm25_index: someone asked for a BM25 instance
2026-03-25 20:04:37,647 [INFO] app.services.embedding_service: embedding service has generated embedding for the query <class 'str'>
2026-03-25 20:04:37,647 [INFO] app.services.embedding_service: first 5 values of the embedding: [-0.03607562184333801, -0.0317964032292366, -0.0024708088021725416, -0.07339930534362793, -0.004712013993412256]
2026-03-25 20:04:37,652 [INFO] app.services.retriever: QUERY EMBEDDING TIME: 284.55591201782227 ms
2026-03-25 20:04:37,654 [INFO] app.services.vector_store: trying to get collection collection_11 from chromadb
2026-03-25 20:04:37,788 [INFO] app.services.retriever: VECTOR SEARCH TIME: 134.45329666137695 ms
2026-03-25 20:04:38,418 [INFO] app.services.bm25_index: Loaded BM25 index for collection 11 with 986 documents
2026-03-25 20:04:38,426 [INFO] app.services.retriever: BM25 Index SEARCH TIME: 636.4927291870117 ms
Batches: 100%|██████████| 2/2 [00:03<00:00,  1.54s

Processed 5/120
Done.


In [7]:
# Aggregate timings
retrieval_all = [r["timings_ms"]["retrieval_ms"] for r in results if "timings_ms" in r]
ttft_all = [r["timings_ms"]["ttft_ms"] for r in results if "timings_ms" in r]
ttft_ar_all = [r["timings_ms"]["ttft_after_retrieval_ms"] for r in results if "timings_ms" in r]
gen_all = [r["timings_ms"]["generation_wall_ms"] for r in results if "timings_ms" in r]
e2e_all = [r["timings_ms"]["e2e_ms"] for r in results if "timings_ms" in r]

print("=== Latency (milliseconds) ===")
summarize_timings("retrieval", retrieval_all)
summarize_timings("TTFT (request to first token)", ttft_all)
summarize_timings("TTFT after retrieval (first token)", ttft_ar_all)
summarize_timings("generation wall (stream total)", gen_all)
summarize_timings("end-to-end (retrieve + stream)", e2e_all)

# Aggregate LLM-judge scores (exclude failed evals)
dims = ["faithfulness", "relevance", "completeness", "citation_accuracy"]
scores: Dict[str, List[float]] = {d: [] for d in dims}
overall: List[float] = []

for r in results:
    ev = r.get("evaluation") or {}
    if not ev or ev.get("error"):
        continue
    for d in dims:
        part = ev.get(d) or {}
        if isinstance(part, dict) and "score" in part:
            try:
                scores[d].append(float(part["score"]))
            except (TypeError, ValueError):
                pass
    if "overall_score" in ev:
        try:
            overall.append(float(ev["overall_score"]))
        except (TypeError, ValueError):
            pass

print("\n=== Answer quality (LLM-as-Judge, 1–5) ===")
for d in dims:
    v = scores[d]
    if not v:
        print(f"{d}: no samples")
        continue
    arr = np.array(v, dtype=float)
    print(
        f"{d}: n={arr.size} mean={arr.mean():.3f} p50={np.percentile(arr, 50):.3f} "
        f"p95={np.percentile(arr, 95):.3f}"
    )
if overall:
    o = np.array(overall, dtype=float)
    print(
        f"overall_score: n={o.size} mean={o.mean():.3f} p50={np.percentile(o, 50):.3f} "
        f"p95={np.percentile(o, 95):.3f}"
    )

failed = sum(
    1
    for r in results
    if "timings_ms" not in r or (r.get("evaluation") or {}).get("error")
)
print(f"\nRows with eval errors or exceptions: {failed}")

=== Latency (milliseconds) ===
retrieval: n=5 mean=2604.8ms p50=1990.1ms p95=4077.5ms max=4143.4ms
TTFT (request to first token): n=5 mean=4862.4ms p50=3887.5ms p95=7074.0ms max=7289.2ms
TTFT after retrieval (first token): n=5 mean=2257.5ms p50=2066.1ms p95=3194.1ms max=3475.0ms
generation wall (stream total): n=5 mean=5749.0ms p50=3899.3ms p95=11743.3ms max=13311.8ms
end-to-end (retrieve + stream): n=5 mean=8353.8ms p50=8042.7ms p95=13828.6ms max=14964.9ms

=== Answer quality (LLM-as-Judge, 1–5) ===
faithfulness: n=3 mean=5.000 p50=5.000 p95=5.000
relevance: n=3 mean=3.000 p50=4.000 p95=4.000
completeness: n=3 mean=3.333 p50=3.000 p95=4.800
citation_accuracy: n=3 mean=5.000 p50=5.000 p95=5.000
overall_score: n=3 mean=3.467 p50=3.200 p95=4.100

Rows with eval errors or exceptions: 2


In [8]:
# Optional: save full results for your resume / reports
out_path = NOTEBOOK_DIR / "evaluation_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "collection_id": COLLECTION_ID,
            "answer_provider": ANSWER_PROVIDER or Config.DEFAULT_LLM_PROVIDER,
            "answer_model": ANSWER_MODEL or Config.DEFAULT_MODEL_NAME,
            "eval_provider": EVAL_PROVIDER or Config.EVALUATOR_MODEL_PROVIDER,
            "eval_model": EVAL_MODEL or Config.EVALUATOR_MODEL_NAME,
            "results": results,
        },
        f,
        indent=2,
        default=str,
    )
print("Wrote", out_path)

Wrote C:\Users\Devendra\opensource\ai-research-assistant\backend\notebooks\evaluation_results.json


In [9]:
# Tear down app context (run when finished)
_app_ctx.pop()
print("App context popped.")

App context popped.
